## Merge And Rebalance Labeled Sources

This notebook builds the final sarcasm detection dataset for modeling. It merges the OpenAI-labeled Kosovo news file with the FLOSSK historic bootstrap dataset, normalizes labels, removes duplicate article text, and writes a 50/50 balanced dataset.

Run this after:

1. `03a_sarcasm_prepare_dataset.ipynb`
2. `03b_flossk_historical_sources_sync.ipynb`

Then rerun `03d_sarcasm_models_and_evaluation.ipynb`.


### Inputs and outputs

Inputs:

- `data/preprocessed_kosovo_news_with_is_sarcasm_v1.csv`
- `data/sarcasm_flossk_historic_bootstrap.csv`

Output:

- `data/preprocessed_kosovo_news_with_is_sarcasm.csv` is the final 50/50 balanced dataset used by `03d_sarcasm_models_and_evaluation.ipynb`.


In [3]:
from pathlib import Path
import pandas as pd

# -----------------------
# CONFIG
# -----------------------
SEED = 42

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"

KOSOVO_LABELED_FILE = DATA_DIR / "preprocessed_kosovo_news_with_is_sarcasm_v1.csv"
FLOSSK_LABELED_FILES = [
    DATA_DIR / "sarcasm_flossk_historic_bootstrap.csv",
]

OUT_BALANCED = DATA_DIR / "preprocessed_kosovo_news_with_is_sarcasm.csv"

# -----------------------
# HELPERS
# -----------------------
def read_csv_robust(path: Path) -> pd.DataFrame:
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin1"):
        try:
            return pd.read_csv(path, engine="python", on_bad_lines="skip", encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(
        path,
        engine="python",
        on_bad_lines="skip",
        encoding="utf-8",
        encoding_errors="replace",
    )


def normalize_label(value):
    if pd.isna(value):
        return pd.NA

    value_norm = str(value).strip().lower()
    positive_values = {"1", "1.0", "yes", "true", "sarcasm", "sarkazem", "sarkazëm"}
    negative_values = {"0", "0.0", "no", "false", "not_sarcasm", "non_sarcasm", "jo"}

    if value_norm in positive_values:
        return 1
    if value_norm in negative_values:
        return 0
    return pd.NA


def normalize_source(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]

    if "content" not in df.columns:
        if "text" in df.columns:
            df["content"] = df["text"]
        elif "body" in df.columns:
            df["content"] = df["body"]
        else:
            raise ValueError(f"{source_name}: missing text column. Expected content/text/body.")

    if "is_sarcasm" not in df.columns:
        if "sarcasm" in df.columns:
            df["is_sarcasm"] = df["sarcasm"]
        elif "labels" in df.columns:
            df["is_sarcasm"] = df["labels"]
        else:
            raise ValueError(f"{source_name}: missing label column. Expected is_sarcasm/sarcasm/labels.")

    if "category" not in df.columns:
        df["category"] = source_name

    if "source" not in df.columns:
        df["source"] = source_name

    df["content"] = df["content"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df["is_sarcasm"] = df["is_sarcasm"].map(normalize_label)
    df["source_dataset"] = source_name

    df = df[(df["content"] != "") & df["is_sarcasm"].isin([0, 1])].copy()
    df["is_sarcasm"] = df["is_sarcasm"].astype(int)

    return df[["content", "category", "source", "source_dataset", "is_sarcasm"]]


def load_source(path: Path, source_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing input file for {source_name}: {path}")
    df = normalize_source(read_csv_robust(path), source_name)
    print(f"{source_name}:", df["is_sarcasm"].value_counts().sort_index().to_dict(), "rows:", len(df))
    return df

# -----------------------
# LOAD + MERGE
# -----------------------
frames = [load_source(KOSOVO_LABELED_FILE, "kosovo_news_labeled_v1")]

for flossk_file in FLOSSK_LABELED_FILES:
    frames.append(load_source(flossk_file, flossk_file.stem))

merged = pd.concat(frames, ignore_index=True)
merged["content_norm"] = merged["content"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

# If duplicate text appears with conflicting labels, keep the sarcasm label first.
merged = (
    merged.sort_values(["content_norm", "is_sarcasm"], ascending=[True, False])
    .drop_duplicates(subset=["content_norm"], keep="first")
    .drop(columns=["content_norm"])
    .sample(frac=1.0, random_state=SEED)
    .reset_index(drop=True)
)

print("merged counts:", merged["is_sarcasm"].value_counts().sort_index().to_dict())
# -----------------------
# 50/50 BALANCE
# -----------------------
sarcasm_df = merged[merged["is_sarcasm"] == 1]
non_sarcasm_df = merged[merged["is_sarcasm"] == 0]

n = min(len(sarcasm_df), len(non_sarcasm_df))
if n == 0:
    raise ValueError("Cannot build a balanced dataset because one class has 0 rows.")

if len(non_sarcasm_df) < len(sarcasm_df):
    print("Warning: fewer non-sarcasm rows than sarcasm rows; balancing to the smaller class size.")

balanced_df = pd.concat(
    [
        sarcasm_df.sample(n=n, random_state=SEED),
        non_sarcasm_df.sample(n=n, random_state=SEED),
    ],
    ignore_index=True,
).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Keep the columns expected by the modeling notebook first.
final_cols = ["content", "category", "is_sarcasm", "source", "source_dataset"]
balanced_df = balanced_df[final_cols]

print("balanced counts:", balanced_df["is_sarcasm"].value_counts().sort_index().to_dict())
print("balanced rows:", len(balanced_df))

balanced_df.to_csv(OUT_BALANCED, index=False, encoding="utf-8")

print("Saved final balanced dataset:", OUT_BALANCED)
balanced_df.head()


kosovo_news_labeled_v1: {0: 119234, 1: 766} rows: 120000
sarcasm_flossk_historic_bootstrap: {0: 600, 1: 600} rows: 1200
merged counts: {0: 119831, 1: 1366}
balanced counts: {0: 1366, 1: 1366}
balanced rows: 2732
Saved final balanced dataset: /Users/bleronaidrizi/Sources/Master_Tema_e_Diplomes/Punimi/Sarcasm-Detection-Albanian-News-Dataset/data/preprocessed_kosovo_news_with_is_sarcasm.csv


,content,category,is_sarcasm,source,source_dataset
0,Deputetja nga radhët e Partisë Demokratike të ...,Lajme,1,GazetaExpress,kosovo_news_labeled_v1
1,Kryeministri Rama ka qenë në Shkodër për proçe...,Ballina;Shqiperi,1,GazetaExpress,kosovo_news_labeled_v1
2,Po zhvillohet në Itali faza e parë e eksperime...,Lajme,0,GazetaExpress,kosovo_news_labeled_v1
3,Shumë fansa e kanë përgëzuar Rina Balajn për “...,Rozë,1,GazetaExpress,kosovo_news_labeled_v1
4,shprehurit artistik. Ajo që e forcon konceptin...,sarcasm_flossk_historic_bootstrap,1,sarcasm_flossk_historic_bootstrap,sarcasm_flossk_historic_bootstrap
